# 레슨 02 — URL 파라미터와 페이지네이션

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/02/%EB%A0%88%EC%8A%A8%2002%20%E2%80%94%20URL%20%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0%EC%99%80%20%ED%8E%98%EC%9D%B4%EC%A7%80%EB%84%A4%EC%9D%B4%EC%85%98.ipynb)

이 레슨은 검색 URL을 문자열이 아니라 구조화된 데이터로 읽는 방법을 다룬다. 학생은 query string을 분해하고 다시 조립하며, 여러 장으로 나뉜 검색 결과를 안전하게 순회한다. 모든 예제는 수업용 합성 HTML fixture를 사용하므로 실제 웹사이트에 반복 요청을 보내지 않는다.

## 학습 목표

1. URL을 scheme, domain, path, query string으로 분해한다.
2. parse_qs, urlencode, urljoin으로 검색 조건과 상대 링크를 안전하게 다룬다.
3. 페이지 번호 규칙을 함수로 분리해 반복 수집 코드를 단순하게 만든다.
4. 페이지네이션 영역의 다음 링크와 현재 페이지 데이터를 구분한다.
5. 여러 페이지에서 모은 결과를 필터링, 집계, CSV 저장까지 연결한다.

---

## 1. URL은 문자열이 아니라 구조다

검색 페이지 URL은 길게 보이지만 실제로는 몇 개의 역할로 나뉜다. 예를 들어 https://example.com/library/search?q=python&category=all&page=2 에서 path는 /library/search이고, query string은 q, category, page 같은 검색 조건을 담는다.

자동화 코드를 만들 때 URL 전체를 문자열로 붙이면 실수하기 쉽다. 검색어에 공백이나 한글이 들어가면 직접 붙인 문자열은 깨질 수 있다. 그래서 파이썬에서는 먼저 URL을 분해하고, 조건은 딕셔너리로 관리한 다음 다시 안전하게 조립한다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def page_filename(page):
    return f'search_page_{page}.html'

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = urlparse(sample_url)
params = parse_qs(parsed.query)
print('path:', parsed.path)
print('query dict:', params)
print('page:', params['page'][0])


> **웹 자동화 안전 한스푼 — query string**
>
> - **뜻**: URL의 물음표 뒤에 붙는 검색 조건이다.
> - **왜 중요한가**: 검색어, 페이지 번호, 필터 조건이 여기 들어가므로 잘못 조립하면 다른 데이터를 가져온다.
> - **수업 기준**: query string은 직접 문자열로 이어 붙이지 않고 urlencode로 만든다.
> - **실수 예시**: q, page 값을 더하기 연산으로 붙이다가 공백과 숫자 처리를 놓친다.

---

## 2. 검색 조건을 딕셔너리로 관리하기

검색 자동화는 보통 같은 URL에 조건만 바꿔 여러 번 실행한다. 조건을 딕셔너리로 두면 검색어, 카테고리, 페이지 번호를 코드 안에서 명확하게 볼 수 있고, CSV에서 읽은 조건도 그대로 넣을 수 있다.

urlencode는 공백과 특수문자를 URL에 맞게 인코딩한다. 학생이 직접 퍼센트 인코딩을 외울 필요가 없다.


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': 'python automation', 'category': 'course', 'page': 1}
url = base_url + '?' + urlencode(query)
print(url)

query['page'] = 2
print(base_url + '?' + urlencode(query))


---

## 3. fixture HTML 읽기

이번 레슨의 검색 결과는 search_page_1.html, search_page_2.html, search_page_3.html 세 파일로 준비되어 있다. 각 파일에는 article.result-card 9개가 들어 있고, 각 카드에는 제목, 상세 링크, 카테고리, 날짜, 조회수가 있다.

실제 사이트를 요청하지 않고 파일 fixture를 쓰는 이유는 수업 중 같은 결과를 안정적으로 재현하기 위해서다. 학생이 반복 실행해도 외부 서버에 부하가 가지 않고, 선생님은 selector 오류와 코드 오류를 구분해서 지도할 수 있다.


In [ ]:
html_text = load_text('search_page_1.html')
soup = BeautifulSoup(html_text, 'html.parser')
heading = soup.select_one('h1').text.strip()
cards = soup.select('article.result-card')
first = cards[0]
print(heading)
print('card count:', len(cards))
print(first.select_one('.title a').text.strip())
print(first['data-category'], first['data-page'])


> **웹 자동화 안전 한스푼 — fixture**
>
> - **뜻**: 수업이나 테스트를 위해 고정해 둔 샘플 데이터다.
> - **왜 중요한가**: 외부 사이트 상태가 바뀌어도 수업 결과가 흔들리지 않는다.
> - **수업 기준**: 1~5강은 실제 사이트 대신 합성 fixture로 selector와 반복 구조를 연습한다.
> - **실수 예시**: fixture에서 충분히 연습하지 않고 실제 사이트를 빠르게 반복 요청한다.

---

## 4. 카드 하나를 레코드로 바꾸기

HTML 카드 하나를 그대로 저장하면 나중에 정렬하거나 필터링하기 어렵다. 자동화 결과는 사람이 읽는 화면에서 파이썬이 다루기 쉬운 딕셔너리로 바꾸는 과정이 필요하다.

여기서는 제목, 카테고리, 페이지 번호, 날짜, 조회수, 상세 URL을 하나의 딕셔너리로 만든다. 조회수는 조회 1,023 같은 문자열이므로 숫자만 남겨 정수로 바꾼다. 링크는 /library/... 형태의 상대 경로이므로 urljoin으로 절대 URL을 만든다.


In [ ]:
def parse_result_card(card, base='https://example.com'):
    link = card.select_one('.title a')
    return {
        'title': link.text.strip(),
        'category': card['data-category'],
        'page': int(card['data-page']),
        'rank': int(card['data-rank']),
        'date': card.select_one('time')['datetime'],
        'views': clean_int(card.select_one('.views').text),
        'url': urljoin(base, link['href']),
        'detail_url': urljoin(base, card.select_one('a.detail')['href']),
    }

record = parse_result_card(first)
print(record)


> **웹 자동화 안전 한스푼 — 상대 URL**
>
> - **뜻**: /library/web-resource-1처럼 도메인 없이 경로만 있는 링크다.
> - **왜 중요한가**: 그대로 저장하면 어느 사이트의 링크인지 알 수 없다.
> - **수업 기준**: 저장 전 urljoin(base_url, href)로 절대 URL을 만든다.
> - **실수 예시**: 상대 경로만 CSV에 저장해 다음 자동화 단계에서 링크를 열 수 없다.

---

## 5. 한 페이지를 리스트로 정리하기

자동화에서 반복 단위가 정해지면 다음 단계는 리스트를 만드는 것이다. 한 페이지 안의 모든 article.result-card를 parse_result_card 함수로 바꾸면 결과는 딕셔너리 리스트가 된다.

이때 len(records)를 먼저 확인하는 습관이 중요하다. 첫 번째 값만 출력하면 selector가 일부만 맞아도 지나칠 수 있지만, 개수를 확인하면 fixture 구조와 코드가 맞는지 빠르게 판단할 수 있다.


In [ ]:
page1_records = [parse_result_card(card) for card in cards]
print('records:', len(page1_records))
print(page1_records[0]['title'], page1_records[0]['views'])
print(page1_records[-1]['title'], page1_records[-1]['views'])


---

## 6. 페이지 번호 규칙을 함수로 분리하기

이번 fixture의 파일명은 search_page_1.html, search_page_2.html, search_page_3.html이다. 페이지 번호가 들어가는 자리를 함수로 분리하면 반복문에서 파일명을 직접 조립하지 않아도 된다.

함수로 분리하는 이유는 단순히 코드가 짧아지기 때문만이 아니다. 실제 사이트에서 페이지 규칙이 바뀌거나 파일명 규칙이 바뀌면 함수 한 곳만 수정하면 된다.


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'

for page in range(1, 4):
    print(page, page_filename(page))


---

## 7. 여러 페이지 순회하기

여러 페이지를 순회할 때는 바깥 반복문이 페이지를 바꾸고, 안쪽 반복문이 카드들을 처리한다. 이 구조를 분명히 이해해야 나중에 페이지네이션이 많은 사이트에서도 무한 반복을 피할 수 있다.

수업에서는 1~3페이지로 고정하지만, 실제 운영 코드에서는 다음 링크 존재 여부, 결과 개수, 최대 페이지 수 같은 중단 기준을 반드시 둔다.


In [ ]:
all_records = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    page_cards = page_soup.select('article.result-card')
    print('page', page, 'cards', len(page_cards))
    for card in page_cards:
        all_records.append(parse_result_card(card))

print('total:', len(all_records))
print(all_records[0]['title'], '->', all_records[-1]['title'])


> **웹 자동화 안전 한스푼 — 페이지네이션 중단 기준**
>
> - **뜻**: 반복 수집을 언제 멈출지 정하는 규칙이다.
> - **왜 중요한가**: 중단 기준이 없으면 같은 요청을 무한히 반복할 수 있다.
> - **수업 기준**: fixture는 1~3페이지로 제한하고, 실제 사이트는 최대 페이지 수나 다음 링크 존재 여부를 확인한다.
> - **실수 예시**: 무한 반복으로 계속 다음 페이지를 요청하고 빈 결과를 확인하지 않는다.

---

## 8. 다음 페이지 링크 읽기

페이지 번호를 직접 증가시키는 방식과 별개로, HTML 안의 페이지네이션 영역을 읽을 수도 있다. nav.pagination a.next는 다음 페이지 링크를 제공한다. 이 링크에서 다시 query string을 읽으면 다음 page 값을 확인할 수 있다.


In [ ]:
next_link = soup.select_one('nav.pagination a.next')
next_href = next_link['href']
next_params = parse_qs(urlparse(next_href).query)
print(next_href)
print('next page:', next_params['page'][0])


---

## 9. 필터링과 집계

수집한 데이터는 저장하기 전에 작은 검증과 요약을 거친다. 예를 들어 조회수 1000 이상인 자료만 골라보거나, 카테고리별 개수를 세면 selector가 의도대로 작동했는지 확인할 수 있다.


In [ ]:
popular = [row for row in all_records if row['views'] >= 1000]
print('popular:', len(popular))
print([row['title'] for row in popular[:5]])

category_counts = {}
for row in all_records:
    category = row['category']
    category_counts[category] = category_counts.get(category, 0) + 1
print(category_counts)


---

## 10. 검색 계획 CSV 읽기

search_targets.csv는 검색어, 카테고리, 최소 조회수, 최대 페이지 수를 담은 계획 파일이다. 실제 업무에서는 검색 조건을 코드 안에 박아두기보다 CSV나 설정 파일로 분리하는 편이 관리하기 쉽다.


In [ ]:
target_rows = list(csv.DictReader(load_text('search_targets.csv').splitlines()))
for row in target_rows:
    print(row['query'], row['category'], row['min_views'], row['max_pages'])


---

## 11. CSV로 저장하기

마지막으로 여러 페이지에서 모은 데이터를 CSV로 저장한다. 저장 전에는 필드 이름을 먼저 정한다. 필드 이름이 일정해야 나중에 엑셀, 구글시트, 데이터 분석 코드에서 같은 구조로 읽을 수 있다.


In [ ]:
fieldnames = ['title', 'category', 'page', 'date', 'views', 'url']
with open('lesson02_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({key: row[key] for key in fieldnames} for row in all_records)
print('saved:', 'lesson02_results.csv', len(all_records))


---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 수집 목적을 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 12. 디버깅 순서

페이지네이션 자동화가 실패했을 때는 selector부터 고치지 않는다. 먼저 조합된 URL이 의도한 조건을 담고 있는지 확인한다. 그다음 HTML을 읽었는지, 반복 단위 개수가 예상과 맞는지, 카드 안에서 필요한 값이 빠지지 않았는지 차례로 확인한다.

수업 중에는 아래 순서를 그대로 말하게 한다.

1. 현재 page 번호와 파일명 또는 URL을 출력한다.
2. HTML 제목 h1을 출력해 올바른 페이지를 읽었는지 확인한다.
3. article.result-card 개수를 출력한다.
4. 첫 카드의 title, href, data-category를 출력한다.
5. 조회수를 정수로 바꾼 뒤 타입을 확인한다.
6. 저장 전 rows 길이와 첫 행의 key 목록을 확인한다.

이 순서를 지키면 학생이 selector를 무작정 바꾸는 시간을 줄일 수 있다. 특히 페이지 반복에서는 첫 페이지 결과가 세 번 들어가는 실수가 자주 나오므로 page_soup을 반복문 안에서 새로 만드는지 확인한다.

## 13. 실제 사이트로 확장하기 전 점검

이번 레슨은 fixture만 사용하지만, 실제 사이트로 확장할 때는 코드보다 운영 기준을 먼저 정해야 한다. 검색 결과가 공개 페이지인지, robots 정책에서 차단하지 않는지, 요청 간격을 둘 수 있는지, 수집한 값을 저장해도 되는지 확인한다. 학생에게는 “코드가 된다”와 “운영해도 된다”가 다르다는 점을 반복해서 설명한다.

robots_sample.txt는 실제 법적 판단을 대신하지 않는다. 수업에서는 허용/차단/지연 요청의 개념을 익히는 샘플로만 사용한다. 실제 서비스에서는 사이트 약관, 관리자 허가, 개인정보 여부를 함께 검토해야 한다.

## 14. 수업 중 확인 질문

- query string에서 q, category, page는 각각 어떤 역할인가?
- 검색어에 공백이 들어갈 때 urlencode가 필요한 이유는 무엇인가?
- article.result-card 대신 a 태그를 반복 단위로 잡으면 어떤 문제가 생기는가?
- 상대 URL을 그대로 저장하면 다음 자동화 단계에서 어떤 정보가 부족한가?
- range(1, 4)가 1, 2, 3을 만든다는 사실을 어디에서 확인할 수 있는가?
- 조회수 문자열을 정수로 바꾸지 않으면 필터링 결과가 왜 틀릴 수 있는가?
- CSV 저장 전에 rows 길이와 fieldnames를 확인하는 이유는 무엇인가?

## 15. 이번 레슨의 완성 기준

학생이 완성해야 하는 것은 단순히 CSV 파일 하나가 아니다. URL 조건을 구조화하고, 페이지 반복 범위를 통제하고, 카드 데이터를 같은 딕셔너리 구조로 맞춘 뒤, 저장 전 간단한 검증을 하는 흐름이다. 이 네 가지가 연결되어야 다음 레슨의 테이블/리스트 데이터 정리로 자연스럽게 넘어갈 수 있다.

---

## 16. 예제 데이터를 읽는 관찰 포인트

search_page 파일들은 일부러 같은 구조를 유지하면서 값만 다르게 만들었다. 학생은 먼저 “무엇이 반복되고 무엇이 달라지는지”를 말해야 한다. 반복되는 것은 article.result-card, .title a, .views, time 태그이고, 달라지는 것은 data-category, data-page, data-rank, 제목, 조회수다.

이 관찰을 먼저 하지 않으면 selector를 외워서 쓰게 된다. 수업에서는 코드를 치기 전에 HTML 일부를 읽고, 반복 단위와 필요한 필드를 표로 정리하게 한다. 이 과정이 있어야 3강에서 table, list, card 구조가 바뀌어도 같은 방식으로 접근할 수 있다.

## 17. 저장 파일을 검토하는 기준

CSV 저장 후에는 파일이 만들어졌다는 사실만 보지 않는다. 첫 줄에 header가 있는지, title/category/views 같은 필드 이름이 일관적인지, 행 수가 예상한 카드 수와 맞는지 확인한다. 이 레슨에서는 3페이지와 페이지당 9개 카드이므로 전체 rows는 27개가 되어야 한다.

필터링 결과는 기준에 따라 달라질 수 있다. 그래서 최소 조회수 기준을 코드 주석이나 결과 요약에 남겨야 한다. 운영자가 다음에 같은 자동화를 실행할 때 기준을 모르면 결과 차이를 오류로 오해할 수 있다.


# 레슨 02 — 실습 문제 정답지

> 교사·관리자 전용. 학생에게 배포 금지.

URL 파라미터와 페이지네이션 실습 문제의 모범 답안이다. 출력값만 확인하지 말고 URL 분해, selector 기준, 타입 변환, 저장 흐름을 같이 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def page_filename(page):
    return f'search_page_{page}.html'

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 정답 — 검색 URL 분해하기


In [ ]:
sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = urlparse(sample_url)
params = parse_qs(parsed.query)
print(parsed.path)
print(params['q'][0], params['category'][0], params['page'][0])


### 왜 이 코드가 정답인지

URL의 path와 query는 역할이 다르다. urlparse로 URL을 구조화해야 path만 따로 확인할 수 있고, parse_qs로 q, category, page를 키로 가진 딕셔너리를 얻는다. query 값은 리스트로 반환되므로 첫 값을 읽는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 2 정답 — 쿼리 문자열 만들기


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': 'python automation', 'category': 'course', 'page': 1}
url = base_url + '?' + urlencode(query)
print(url)


### 왜 이 코드가 정답인지

urlencode는 딕셔너리의 키와 값을 URL query string 규칙에 맞춰 변환한다. 검색어에 공백이 있어도 안전하게 인코딩되므로 문자열을 직접 이어 붙이는 방식보다 운영 자동화에 적합하다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 3 정답 — 첫 페이지 HTML 읽기


In [ ]:
html_text = load_text('search_page_1.html')
soup = BeautifulSoup(html_text, 'html.parser')
print(soup.select_one('h1').text.strip())


### 왜 이 코드가 정답인지

load_text는 코랩과 로컬 경로 차이를 숨긴다. BeautifulSoup 객체로 바꾼 뒤 select_one으로 페이지 제목 하나를 안정적으로 가져올 수 있다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 4 정답 — 결과 카드 개수 세기


In [ ]:
cards = soup.select('article.result-card')
print('cards:', len(cards))


### 왜 이 코드가 정답인지

페이지 전체에서 같은 구조가 반복될 때는 select로 리스트를 받는다. article.result-card는 다른 링크나 pagination을 제외하고 검색 결과 카드만 잡는 selector다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 5 정답 — 첫 카드 제목과 href 읽기


In [ ]:
first = cards[0]
title = first.select_one('.title a').text.strip()
href = first.select_one('.title a')['href']
print(title)
print(href)


### 왜 이 코드가 정답인지

카드 하나 안에서 다시 .title a를 찾으면 다른 카드의 링크와 섞이지 않는다. 텍스트는 정리하고, 링크 경로는 href 속성으로 읽는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 6 정답 — 상대 URL을 절대 URL로 바꾸기


In [ ]:
absolute = urljoin('https://example.com', href)
print(absolute)


### 왜 이 코드가 정답인지

상대 링크는 저장만 해서는 어느 도메인의 경로인지 알 수 없다. urljoin은 기준 URL과 상대 경로를 올바르게 합쳐 다음 자동화 단계에서 바로 열 수 있는 URL을 만든다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 7 정답 — 카드 하나를 딕셔너리로 만들기


In [ ]:
item = {
    'title': first.select_one('.title a').text.strip(),
    'category': first['data-category'],
    'date': first.select_one('time')['datetime'],
    'views': clean_int(first.select_one('.views').text),
    'url': urljoin('https://example.com', first.select_one('.title a')['href']),
}
print(item)


### 왜 이 코드가 정답인지

웹 화면의 값은 태그 텍스트, HTML 속성, 문자열 숫자가 섞여 있다. 이 답안은 각 값의 위치에 맞는 읽기 방식을 사용하고, 저장하기 좋은 딕셔너리 구조로 통일한다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 8 정답 — 한 페이지 결과 리스트 만들기


In [ ]:
results = []
for card in cards:
    results.append({
        'title': card.select_one('.title a').text.strip(),
        'category': card['data-category'],
        'page': int(card['data-page']),
        'views': clean_int(card.select_one('.views').text),
    })
print(results[0])
print(len(results))


### 왜 이 코드가 정답인지

한 페이지 안의 카드들을 같은 딕셔너리 구조로 만들면 이후 필터링과 저장이 쉬워진다. data-page는 문자열 속성이므로 정수로 변환한다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 9 정답 — 페이지 번호로 파일명 만들기


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'

for page in [1, 2, 3]:
    print(page_filename(page))


### 왜 이 코드가 정답인지

페이지 파일명 규칙을 함수로 분리하면 여러 페이지 순회 코드에서 파일명 문자열을 반복 작성하지 않아도 된다. 규칙이 바뀌어도 함수 한 곳만 수정하면 된다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 10 정답 — 3페이지 전체 순회하기


In [ ]:
all_results = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        all_results.append(card.select_one('.title a').text.strip())
print(len(all_results))
print(all_results[-1])


### 왜 이 코드가 정답인지

바깥 반복문은 페이지를 바꾸고 안쪽 반복문은 카드 단위를 처리한다. 3개 파일에 각각 9개 카드가 있으므로 전체 제목 수로 페이지 순회를 확인할 수 있다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 11 정답 — 다음 페이지 링크 찾기


In [ ]:
next_href = soup.select_one('a.next')['href']
next_query = parse_qs(urlparse(next_href).query)
print(next_href)
print(next_query['page'][0])


### 왜 이 코드가 정답인지

페이지네이션 링크는 화면에 보이는 숫자뿐 아니라 다음 페이지로 가는 URL을 제공한다. href에서 query string을 다시 분해하면 다음 page 값을 코드로 판단할 수 있다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 12 정답 — 조회수 1000 이상 필터링


In [ ]:
rich = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        views = clean_int(card.select_one('.views').text)
        if views >= 1000:
            rich.append(card.select_one('.title a').text.strip())
print(rich)


### 왜 이 코드가 정답인지

HTML의 조회수는 사람이 읽는 문자열이다. 숫자 비교를 하려면 clean_int로 쉼표와 한글을 제거해 정수로 바꾼 뒤 기준값 1000과 비교해야 한다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 13 정답 — 카테고리별 개수 세기


In [ ]:
counts = {}
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        key = card['data-category']
        counts[key] = counts.get(key, 0) + 1
print(counts)


### 왜 이 코드가 정답인지

카테고리는 카드의 data-category 속성에 들어 있다. 딕셔너리 집계는 기존 값이 없을 때 0에서 시작해 1씩 더하는 방식으로 구현한다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 14 정답 — targets CSV 읽기


In [ ]:
target_rows = list(csv.DictReader(load_text('search_targets.csv').splitlines()))
for row in target_rows:
    print(row['query'], row['max_pages'])


### 왜 이 코드가 정답인지

CSV를 DictReader로 읽으면 query, category, min_views, max_pages 컬럼을 이름으로 접근할 수 있다. 코드 안에 검색 조건을 고정하지 않고 파일로 분리하는 운영 패턴이다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 15 정답 — 검색 결과 CSV 저장하기


In [ ]:
rows = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        rows.append({
            'title': card.select_one('.title a').text.strip(),
            'category': card['data-category'],
            'views': clean_int(card.select_one('.views').text),
        })
with open('lesson02_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'category', 'views'])
    writer.writeheader()
    writer.writerows(rows)
print('saved:', 'lesson02_results.csv', len(rows))


### 왜 이 코드가 정답인지

여러 페이지에서 모은 결과를 같은 키 구조로 만들고 DictWriter로 저장한다. 헤더를 먼저 쓰고 전체 rows를 저장해야 CSV를 다시 열었을 때 컬럼 의미가 유지된다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 URL 처리 함수가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 교사용 종합 채점 기준

이 레슨은 문법 암기보다 자동화 흐름을 보는 단원이다. 학생 답안이 출력값을 맞히더라도 URL 조립, selector 기준, 타입 변환, 저장 구조가 불안정하면 부분 감점한다. 반대로 출력 문구가 조금 달라도 중간 데이터 구조가 명확하면 통과로 볼 수 있다.

### URL 처리 기준

문제 1과 2에서는 urlparse, parse_qs, urlencode를 구분해서 쓰는지 확인한다. query string을 split으로만 처리한 답안은 예제에서는 동작할 수 있지만 검색어에 공백이나 특수문자가 들어가면 깨진다. 학생이 왜 표준 함수를 쓰는지 설명할 수 있어야 한다.

### selector 기준

문제 3~5에서는 반복 단위와 내부 selector를 구분하는지 확인한다. article.result-card는 카드 전체를 대표하고, .title a는 카드 내부 제목 링크를 대표한다. 전체 soup에서 매번 첫 링크만 찾는 답안은 반복문으로 확장했을 때 틀리므로 반드시 card 내부에서 select_one을 호출하게 지도한다.

### 데이터 변환 기준

문제 6~8에서는 사람이 보는 문자열을 저장 가능한 값으로 바꾸는지 확인한다. 조회수는 정수, 페이지 번호는 정수, 날짜는 datetime 속성 문자열, 링크는 절대 URL로 정리되어야 한다. 이 변환이 빠지면 이후 필터링이나 CSV 활용이 불안정해진다.

### 페이지 반복 기준

문제 9~11에서는 반복 범위와 중단 기준을 확인한다. range(1, 4)는 1~3페이지를 의미한다. 학생이 range 끝값을 포함한다고 착각하면 3페이지를 빠뜨린다. 다음 링크를 읽는 문제에서는 a.next href를 읽은 뒤 query string에서 page 값을 다시 분해해야 한다.

### 저장 기준

문제 12~15에서는 필터링 기준과 CSV 헤더를 본다. rows에 들어가는 딕셔너리 key와 DictWriter의 fieldnames가 일치해야 한다. 헤더 없이 문자열로 직접 저장하는 답안은 운영자가 다시 읽기 어렵기 때문에 감점한다.

## 수업 중 피드백 문장 예시

- “지금 선택한 단위가 카드 전체인지 카드 안의 링크인지 먼저 말해보자.”
- “이 값은 사람이 읽는 문자열이니, 비교 전에 숫자로 바꿔야 한다.”
- “페이지 반복은 파일명 함수가 맞아야 전체가 맞는다.”
- “CSV는 다음 사람이 다시 여는 파일이라 헤더가 있어야 한다.”

---

## 문제별 추가 확인 질문

문제 1: parse_qs 결과의 값이 리스트인 이유를 설명할 수 있는가? 같은 key가 여러 번 나올 수 있기 때문이다.

문제 2: urlencode를 쓰지 않고 직접 문자열을 붙이면 어떤 검색어에서 깨질 수 있는가? 공백, 한글, 특수문자가 들어간 검색어다.

문제 3: load_text를 쓰는 이유는 무엇인가? 코랩에서는 raw GitHub URL, 로컬에서는 data 폴더를 읽도록 경로 차이를 숨기기 위해서다.

문제 4: article.result-card 개수가 0이면 먼저 무엇을 확인해야 하는가? HTML 파일을 제대로 읽었는지와 class 이름이 맞는지 확인한다.

문제 5~7: title, href, data-category는 각각 텍스트와 속성 중 어디에 있는가? 이 구분을 못 하면 값은 보이는데 저장 구조가 틀어진다.

문제 8~10: 반복문이 페이지별로 새 soup을 만드는지 확인한다. 같은 soup을 계속 쓰면 첫 페이지 결과가 반복된다.

문제 11~15: 저장 전 rows 길이, 첫 행 key, CSV fieldnames를 확인한다. 이 세 가지가 맞으면 대부분의 저장 오류를 미리 잡을 수 있다.

---

## 심화 채점 기준

심화 풀이에서는 새로 HTML을 다시 읽기보다 이미 만든 rows 또는 all_rows를 재사용하는지 확인한다. 데이터를 한 번 구조화한 뒤 필터링과 정렬을 반복 적용하는 습관이 중요하다. 같은 HTML을 불필요하게 여러 번 읽어도 결과는 맞을 수 있지만, 운영 자동화에서는 느리고 관리하기 어렵다.


# 레슨 02 — 최종 미션 모범 답안

> 교사·관리자 전용. 학생에게 배포 금지.

이 답안은 URL 파라미터, 페이지 반복, 상대 URL 변환, 필터링, CSV 저장을 한 흐름으로 묶는다.

## 모범 답안


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def page_filename(page):
    return f'search_page_{page}.html'

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_result_card(card, base='https://example.com'):
    link = card.select_one('.title a')
    return {
        'title': link.text.strip(),
        'category': card['data-category'],
        'page': int(card['data-page']),
        'rank': int(card['data-rank']),
        'date': card.select_one('time')['datetime'],
        'views': clean_int(card.select_one('.views').text),
        'url': urljoin(base, link['href']),
        'detail_url': urljoin(base, card.select_one('a.detail')['href']),
    }

target_rows = list(csv.DictReader(load_text('search_targets.csv').splitlines()))
all_rows = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        all_rows.append(parse_result_card(card))

min_views_by_category = {row['category']: int(row['min_views']) for row in target_rows}
filtered_rows = [row for row in all_rows if row['views'] >= min_views_by_category.get(row['category'], 0)]

fieldnames = ['title', 'category', 'page', 'rank', 'date', 'views', 'url', 'detail_url']
with open('lesson02_all_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

with open('lesson02_filtered_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(filtered_rows)

category_summary = {}
for row in all_rows:
    item = category_summary.setdefault(row['category'], {'count': 0, 'views': 0})
    item['count'] += 1
    item['views'] += row['views']

for category, item in category_summary.items():
    print(category, item['count'], round(item['views'] / item['count'], 1))
print('all:', len(all_rows))
print('filtered:', len(filtered_rows))


## 왜 이 답안이 기준을 충족하는가

- 검색 조건과 페이지 파일명을 코드에서 분리해 다음 실행 때 수정 지점이 분명하다.
- 카드 하나를 딕셔너리로 바꾸는 함수를 만들어 전체 페이지 반복에서도 같은 구조를 유지한다.
- 상대 링크를 절대 URL로 변환해 CSV만 열어도 이동 가능한 링크가 남는다.
- 저장 전 필터링 기준을 search_targets.csv에서 읽어 운영자가 조건을 파일로 조정할 수 있다.

## 채점 메모

- 학생 답안이 결과를 맞히더라도 URL 변환, 조회수 정수 변환, CSV 헤더가 빠졌으면 감점한다.
- 실제 사이트 URL을 직접 반복 요청한 답안은 이 레슨 기준에서 실패로 본다.
- 필터링 기준을 코드에 하드코딩했더라도 기본 동작은 인정하되, 검색 계획 CSV를 읽지 않았다면 운영 자동화 관점에서 감점한다.

---

## 운영 해설

이 최종 답안은 검색 계획 파일과 페이지 fixture를 분리해 둔다. 운영자가 검색 조건을 바꾸고 싶을 때는 search_targets.csv만 수정하면 되고, 카드 파싱 로직은 유지된다. 실제 사이트로 확장할 때도 이 구조를 유지하면 요청 대상, 반복 범위, 저장 결과를 따로 검토할 수 있다.

학생 답안을 볼 때는 filtered_rows의 개수보다 필터링 기준이 어디에서 왔는지를 먼저 확인한다. 기준을 코드 안에 숫자로 고정해도 예제는 맞을 수 있지만, 운영 자동화에서는 조건을 파일이나 설정으로 분리하는 편이 재사용성이 높다.


# 레슨 02 — 교사 가이드

## 학습 목표 (학생용보다 더 상세)

학생이 URL을 단순 문자열이 아니라 구조화된 데이터로 설명하도록 지도한다. urlparse, parse_qs, urlencode, urljoin의 역할을 구분하고, 페이지네이션 반복문에서 바깥 반복과 안쪽 반복의 책임을 말로 설명할 수 있어야 한다.

## 2시간 타임라인 (분 단위)

- 0~10분: 1강 복습, HTML fixture를 쓰는 이유 확인.
- 10~25분: URL path/query string 분해 시연.
- 25~40분: urlencode로 검색 조건 조립 실습.
- 40~60분: 첫 페이지 카드 selector 찾기.
- 60~75분: 카드 하나를 딕셔너리로 변환.
- 75~95분: 1~3페이지 반복 수집.
- 95~110분: 필터링, 카테고리 집계, CSV 저장.
- 110~120분: 최종 미션 안내와 안전 규칙 정리.

## 사전 준비물

- Colab이 GitHub 노트북을 열 수 있는지 확인한다.
- search_page_1.html부터 search_page_3.html, search_targets.csv, robots_sample.txt가 data 폴더에 있는지 확인한다.
- 학생에게 실제 사이트에 반복 요청하지 않는다는 기준을 먼저 설명한다.

## 핵심 개념 강조 포인트 (수업 진행 시 강조할 부분)

URL query string은 검색 조건을 담는 데이터다. 학생이 URL 전체를 외우려 하면 이후 자동화가 불안정해진다. 조건은 딕셔너리로 관리하고, 문자열 조립은 표준 함수에 맡긴다는 원칙을 반복해서 확인한다.

페이지네이션은 페이지 번호 증가와 카드 반복 처리가 분리되어야 한다. 이 구조가 섞이면 첫 페이지만 반복하거나 마지막 페이지를 빠뜨리는 문제가 자주 나온다.

## 학생이 자주 막히는 지점과 대처법

- parse_qs 결과가 리스트라는 점을 놓친다. params['page'][0] 형태를 직접 출력하게 한다.
- range(1, 3)으로 3페이지를 빠뜨린다. range의 끝값은 포함되지 않는다는 점을 다시 묻는다.
- 조회수 문자열을 숫자로 바꾸지 않는다. clean_int('조회 1,023') 결과를 먼저 확인시킨다.
- 상대 링크를 그대로 저장한다. CSV에 저장된 링크만 보고 접속 가능한지 묻는다.

## 실습 문제 채점 포인트 (문제별)

문제 1~2는 URL 구조와 query 조립을 본다. 문제 3~5는 fixture 로딩과 selector 정확도를 본다. 문제 6~8은 상대 URL, 속성 읽기, 딕셔너리 구조를 본다. 문제 9~11은 페이지 반복과 다음 링크 해석을 본다. 문제 12~15는 필터링, 집계, CSV 저장까지 이어지는지 본다.

## 최종 미션 평가 루브릭 (필수 / 보너스 / 감점)

필수는 전체 페이지 순회, 카드 필드 추출, 조회수 정수 변환, 절대 URL 변환, CSV 저장이다. 보너스는 카테고리별 평균 조회수와 다음 페이지 링크 해석 함수다. 실제 사이트를 직접 반복 요청하거나 학생 노트북에 정답 코드를 붙여 넣은 경우는 감점한다.

## 다음 레슨과의 연결고리

3강은 HTML 테이블과 리스트 데이터를 정리한다. 2강에서 만든 반복 단위를 딕셔너리로 변환한다는 흐름이 그대로 이어진다. 페이지 카드가 테이블 행으로 바뀔 뿐, selector를 찾고 같은 구조로 저장한다는 원칙은 동일하다.


## 수업 중 추가 질문

- URL에 있는 page 값과 HTML 카드의 data-page 값은 왜 둘 다 확인해야 할까?
- 검색어가 한글일 때 직접 문자열을 붙이면 어떤 문제가 생길 수 있을까?
- 다음 페이지 링크가 없어지는 마지막 페이지에서는 어떤 조건으로 반복을 멈춰야 할까?
- CSV에 절대 URL을 남기면 운영자가 어떤 이점을 얻을까?

## 보충 설명 포인트

학생이 빠르게 푸는 경우 search_targets.csv의 min_views를 바꾸고 결과 개수가 어떻게 달라지는지 확인시킨다. 반대로 막히는 학생은 최종 저장보다 cards 개수와 첫 카드 딕셔너리 만들기까지를 우선 통과시킨다.

이 레슨의 핵심은 “여러 페이지를 읽었다”가 아니라 “반복 가능한 검색 자동화 구조를 만들었다”는 점이다. URL 조건, 페이지 반복, 카드 파싱, CSV 저장이 분리되어 있으면 다음 주에도 같은 코드를 다시 실행할 수 있다.


## 난이도 조절

빠른 학생에게는 search_targets.csv의 min_views 값을 바꿔 필터링 결과가 달라지는지 확인하게 한다. 추가로 카테고리별 평균 조회수를 계산하게 하면 데이터 분석 코스와도 연결된다.

느린 학생에게는 최종 CSV 저장까지 요구하기보다 문제 1~8을 먼저 안정화한다. URL 분해, 첫 페이지 카드 선택, 카드 하나를 딕셔너리로 만드는 흐름만 잡혀도 다음 레슨을 따라갈 수 있다.

## 수업 종료 전 확인

마지막 5분에는 학생에게 “이 코드를 실제 사이트에 바로 쓰면 안 되는 이유”를 한 문장으로 쓰게 한다. 서버 부하, 약관, 개인정보, 요청 간격 중 하나라도 언급하면 안전 기준을 이해한 것으로 본다.
